In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
STEMS = {'vocals.wav','other.wav','bass.wav','drums.wav'} 
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
SONG_INDEX = ''  

# NOISE DATASET
root_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/'
csv_file = '/meta/esc50.csv'
audio_folder = '/audio/'


SR = 22050
DURATION = 30
print("✅")

# Classical ML Baseline

* Clean and preprocess audio stems.
* Convert audio to numerical features (MFCCs, Spectrograms).
* Train Logistic Regression, Naive Bayes, or boosting models (CatBoost, LightGBM, XGBoost).
* Evaluate using Macro F1 and log results in W&B.

# Defined functions

In [ ]:
def build_dataset(root_dir, val_split= None, seed=42):
    """
        This takes root dirstory and load paths of all stem files as a dictionary.
        Returns a Dictionary.
    """
    # Initialize empty dictionaries
    song_list = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    total_stem_file = 0
    for g in GENRES:
        folder_path = root_dir + '/' + g
        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            for i in range(0,100):
                song_folder = g + '.' + f"{i:05d}"
                for s in STEMS:
                    stem_file_path = folder_path + '/' + song_folder + '/' + s
                    if os.path.exists(stem_file_path):
                        total_stem_file += 1
                        song_list[g][s.replace('.wav', '')].append(stem_file_path)
                    else:
                        print(f"Stem file {s} not exists !")
            
        else:
            print(f"Folder '{g}' not exists !")

    print("✅ Total number of stems music file loaded successfully : ", total_stem_file)
    
    if val_split :
        train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
        val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
        def add_to_dict(train_dataset,val_dataset,song_list):
            for g in GENRES:
                for s in STEM_KEYS:
                    for k in index[0:83]:
                        train_dataset[g][s].append(song_list[g][s][k])
                    for k in index[83:]:
                        val_dataset[g][s].append(song_list[g][s][k])
        add_to_dict(train_dataset,val_dataset,song_list)
    
        return train_dataset, val_dataset , song_list
    return song_list

def noise_file_information(root_dir=None,audio_folder=None,csv_file=None):
    """
     It will extract all noise audio information and store it in a DataFrame for later analysis.
     Example : {
         audio_file,
         sample_rate,
         total_length,
         duration,
         size,
         total_silence_duration,
         max_silence,
         max_silence_type
    }
    Returns a DataFrame.
    """
    df_noise = None
    _df = None
    
    if csv_file:
        df_noise = pd.read_csv(root_dir+csv_file)
        return df_noise
    if audio_folder:
        path_to_folder = root_dir+audio_folder
        _df = pd.DataFrame(
            index=range(2000),
            columns=[
                 "audio_file",
                 "category",
                 "sample_rate",
                 "total_length",
                 "duration",
                 "size",
                 "total_silence",
                 "max_silence",
                 "max_silence_type"
            ])
        
        j = 0
        for i in df_noise.index:
            filename = df_noise.loc[i]['filename']
            category = df_noise.loc[i]['category']
            path = path_to_folder+filename
            y,sr = lb.load(path = path,sr=None)    # taking original sr and duration
            duration = lb.get_duration(y=y,sr=sr)
            total_length = sr*duration
            size = os.path.getsize(path)
            max_silence,total_silence,max_silence_type = get_silence_information(y,sr,duration,TOP_DB=20)
            _df.loc[j] = [filename,category,sr,total_length,duration,size,total_silence,max_silence,max_silence_type]
            j = j + 1

        return _df
    

def create_single_track(p1,p2,p3,p4):
    """
        This function adding different stems into single track.
        Making all track of equal duration.
    """
    SR = 22050
    DURATION = 30
    LENGTH = SR*DURATION

    def pad(y):
        if len(y) < LENGTH: 
            padding = LENGTH  - len(y) 
            y = np.pad(y, (0, padding), mode='constant') 
        else: 
            y = y[:LENGTH]
        return y
    
    y1,sr1 = lb.load(path=p1,sr=SR)
    y2,sr2 = lb.load(path=p2,sr=SR)
    y3,sr3 = lb.load(path=p3,sr=SR)
    y4,sr4 = lb.load(path=p4,sr=SR)

    mix = pad(y1) + pad(y2) + pad(y3) + pad(y4)
    duration = lb.get_duration(y=mix,sr=SR)
    
    max_amplitude_val = np.max(np.abs(mix))
    
    if(max_amplitude_val > 0):
        track = mix/max_amplitude_val
    else:
        return mix,SR,duration
    return track,SR,duration


def stem_recombination(genre):  
    """
        This function randomly pick different stems from same genre and makes a mashup.
        RETURN : single music track (Combination of stems files of different songs from same genre)
    """
    global sl
    paths = []
    for s in STEM_KEYS:
        p = random.choice(sl[genre][s])
        #print(p)
        paths.append(p)
    mashup,SR,duration = create_single_track(*paths)
    return mashup,SR,duration
    
def load_noise(noise_files):
    """
        It will take noise file names and return a list of noise audio as time-domain.
    """
    global root_dir,audio_folder
    SR = 22050
    DURATION = 5
    LENGTH = SR*DURATION
    
    count_paths = len(noise_files)
    noise = []
    for file in noise_files:
        path = root_dir+audio_folder+file
        y,sr = lb.load(path=path,sr=SR)
        noise.append(y)
    return noise

def add_noise(mashup):
    """
        It will take recombination music and add noise at random places.
    """
    global noise_df
    num_insertions = random.choice([4,5]) 
    noise_files = random.choices(noise_df['filename'],k=num_insertions)
    
    noise = load_noise(noise_files)
    max_start = 22050*(30-5)  # as sample rate for all is 22050 and noise duration is 5sec and audio duration is 30sec.
    
    positions = random.sample(range(0, max_start), num_insertions)
    #print(num_insertions, noise_files, max_start,np.array(positions)/22050)

    output = mashup.copy()
    for i,pos in enumerate(positions):
        output[pos:pos+len(noise[i])] += noise[i]
    output = output / np.max(np.abs(output))
    return output

# Data augmentation function
def data_augmentation(g=None,sample_count=1000):
    """
        Creating actual noisy mashup.
    """
    if g is None:
        return 
    mashup = []
    total_size = 0 #Mb

    print(g)
    # Directory path
    dir_path = f"/kaggle/working/{g}"
    
    # Create directory if it doesn't exist
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"Directory created at: {dir_path}")
    except Exception as e:
        print(f"Error creating directory: {e}")
    
    for i in tqdm(range(0,sample_count),desc="Creating noisy mashup"):
        mashup,SR,duration = stem_recombination(g)
        noisy_mashup = add_noise(mashup)
        sf.write(f"{dir_path}/mashup_{i}.wav", noisy_mashup, SR)
        total_size += os.path.getsize('/kaggle/working/mashup_0.wav')/(1024*1024)
    print(f"🤺 {total_size} Mb of memory is used for storing augmented music for `{g}` genre.")

def extract_features_for_ml(track_arr,sr,g=None):
    """
        For ML feature extraction.
        Take a noisy music track and extract tabular features.
    """
    tempo = lf.tempo(y=track_arr,sr=sr).item()
    harm, perc = lb.effects.hpss(track_arr)
    harmony_mean = np.mean(harm) 
    harmony_var = np.var(harm)

    perceptr_mean = np.mean(perc) 
    perceptr_var = np.var(perc)
    
    rms = lf.rms(y=track_arr)
    rms_mean = np.mean(rms)
    rms_var = np.var(rms)
    
    chroma_stft = lf.chroma_stft(y=track_arr,sr=sr)
    chroma_stft_mean = np.mean(chroma_stft)
    chroma_stft_var = np.var(chroma_stft)
    
    spectral_centroid = lf.spectral_centroid(y=track_arr,sr=sr)
    spectral_centroid_mean = np.mean(spectral_centroid)
    spectral_centroid_var = np.var(spectral_centroid)


    spectral_bandwidth = lf.spectral_bandwidth(y=track_arr,sr=sr)
    spectral_bandwidth_mean = np.mean(spectral_bandwidth)
    spectral_bandwidth_var = np.var(spectral_bandwidth)

    spectral_rolloff = lf.spectral_rolloff(y=track_arr,sr=sr)
    spectral_rolloff_mean = np.mean(spectral_rolloff)
    spectral_rolloff_var = np.var(spectral_rolloff)

    zcr = lf.zero_crossing_rate(y=track_arr)
    zcr_mean = np.mean(zcr)
    zcr_var = np.var(zcr)

    mfccs = lf.mfcc(y=track_arr, sr=sr)
    mfcc_means = np.mean(mfccs,axis=1)
    mfcc_vars = np.var(mfccs,axis=1)

    features = {
        "tempo" : tempo,
        "harmony_mean" : harmony_mean,
        "harmony_var" : harmony_var,
        "perceptr_mean" : perceptr_mean,
        "perceptr_var" : perceptr_var,
        "rms_mean" : rms_mean,
        "rms_var" : rms_var,
        "chroma_stft_mean" : chroma_stft_mean,
        "chroma_stft_var" : chroma_stft_var,
        "spectral_centroid_mean" : spectral_centroid_mean,
        "spectral_centroid_var" : spectral_centroid_var,
        "spectral_bandwidth_mean" : spectral_bandwidth_mean,
        "spectral_bandwidth_var" : spectral_bandwidth_var,
        "spectral_rolloff_mean" : spectral_rolloff_mean,
        "spectral_rolloff_var" : spectral_rolloff_var,
        "zcr_mean" : zcr_mean,
        "zcr_var" : zcr_var
        
    }
    for i in range(0,20):
        features[f'mfcc{i+1}_mean'] = mfcc_means[i]
        features[f'mfcc{i+1}_var'] = mfcc_vars[i]
    if g:
        features["label"] = g
    return features

def create_features_from_dataset(features_name):
    global SR
    ml_df = pd.DataFrame(
        index=range(1000),
        columns= features_name
    )
    genres = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock']
    data_paths = [
        '/kaggle/input/datasets/akashkumbhakar/blues/',
        '/kaggle/input/datasets/akashkumbhakar/classical/',
        '/kaggle/input/datasets/akashkumbhakar/country/',
        '/kaggle/input/datasets/akashkumbhakar/disco/',
        '/kaggle/input/datasets/akashkumbhakar/hiphop/',
        '/kaggle/input/datasets/akashkumbhakar/jazz-genre/',
        '/kaggle/input/datasets/akashkumbhakar/metal/',
        '/kaggle/input/datasets/akashkumbhakar/pop-genre/',
        '/kaggle/input/datasets/akashkumbhakar/reggae-genre/',
        '/kaggle/input/datasets/akashkumbhakar/rock-genre/'
    ]
    j = 0
    for data_path,g in zip(data_paths,genres):
        print(data_path,g)
        for i in tqdm(range(0,100),desc=f"Extraction for `{g}` genre ..."):
            path = data_path + f"mashup_{i}.wav"
            y,sr = lb.load(path,sr=SR)
            f = extract_features_for_ml(y,sr,g=g)
            ml_df.loc[j] = list(f.values())
            j = j + 1

    return ml_df

def create_features_from_testdataset(test_data_path,test_df,features_name):
    global SR
    test_features_df = pd.DataFrame(
        index = range(test_df.shape[0]),
        columns = features_name
    )
    k = 0
    for idx,file in tqdm(zip(test_df['id'],test_df['filename']), desc="Creating test data ...",total=len(test_df)):
        path = test_data_path + file
        y,sr = lb.load(path,sr=SR)
        features = extract_features_for_ml(y,sr)
        values = [idx] + list(features.values())
        test_features_df.loc[k] = values
        k = k + 1
        
    return test_features_df

print("✅ Defined successfully.")

# ML feature extraction for training

In [ ]:
sl = build_dataset(DATA_ROOT)

In [ ]:
# TESTING 
mash,sr,d = stem_recombination('pop')
f = extract_features_for_ml(mash,sr,g='pop')    
features_name = list(f.keys())
print("Total features : ", len(features_name))

In [ ]:
# Features extract from data
ml_df = create_features_from_dataset(features_name)
print("✅ Features extracted from dataset.")    

In [ ]:
# Storing to kaggle hub
handle = f'akashkumbhakar/features-1000'
local_dataset= f'/kaggle/working/music_1000.csv'

# Create a new dataset
dataset_url = kagglehub.dataset_upload(handle, local_dataset)

# ML model training

In [ ]:
ml_df.head(1)

In [ ]:
# Spliting to train and test 
from sklearn.model_selection import train_test_split
cols = list(ml_df.columns)
x_t,x_v,y_t,y_v = train_test_split(ml_df[cols[0:57]],ml_df['label'],test_size=0.2, stratify=ml_df['label'],random_state=0)
print(x_t.shape,x_v.shape,y_t.shape,y_v.shape)

In [ ]:
# Logistic Regression

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix,classification_report,ConfusionMatrixDisplay

ss = StandardScaler()
x_tt = ss.fit_transform(x_t)
x_vt = ss.transform(x_v)


lr = LogisticRegression()
lr.fit(x_tt,y_t)
print("✅")

In [ ]:
# Predict
y_p = lr.predict(x_vt)
macro_f1 = f1_score(y_v,y_p,average='macro')
print("Macro F1 : ", macro_f1)

cm = confusion_matrix(y_v,y_p,labels=genres)
cr = classification_report(y_v,y_p,labels=lr.classes_)
print(cr)

disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=lr.classes_)
disp.plot()
plt.show()

# Test Set creation

In [ ]:
test_data_path = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/'
test_df = pd.read_csv(test_data_path + 'test.csv')

In [ ]:
test_features_name = ['id'] + features_name[:-1]
test_feature_df = create_features_from_testdataset(test_data_path,test_df,test_features_name)

In [ ]:
test_feature_df.head()

In [ ]:
test_feature_df.to_csv('test_music_features.csv')

In [ ]:
# Storing to kaggle hub
handle = f'akashkumbhakar/test-features'
local_dataset= f'/kaggle/working/test_music_features.csv'

# Create a new tabular test dataset
dataset_url = kagglehub.dataset_upload(handle, local_dataset)